# Da She's Voice Engine — Colab GPU
GPU-accelerated TTS using XTTS v2. Falls back to alternative models if XTTS fails.

In [ ]:
# 1. Install TTS (try multiple strategies)
import subprocess, sys, importlib

def install(pkg, desc):
    print(f"Trying: {desc}...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True, timeout=300)
    if r.returncode == 0:
        print(f"  OK")
        return True
    print(f"  Failed: {r.stderr[-200:]}")
    return False

# Strategy 1: Latest from PyPI
if install("TTS", "TTS (PyPI)"):
    pass
# Strategy 2: Older version with broader compat
elif install("TTS==0.22.0", "TTS v0.22.0"):
    pass
# Strategy 3: From GitHub (dev branch)
elif install("git+https://github.com/coqui-ai/TTS.git", "TTS (GitHub dev)"):
    pass
else:
    print("All TTS install strategies failed.")

install("flask", "flask")
install("flask-cors", "flask-cors")

try:
    import TTS
    print(f"TTS version: {TTS.__version__}")
except:
    print("TTS not available")

In [ ]:
# 2. Download speaker sample
import requests
r = requests.get("https://qwert.crousia.com/speaker.wav", timeout=30)
r.raise_for_status()
with open("speaker.wav", "wb") as f: f.write(r.content)
print(f"Downloaded: {len(r.content)} bytes")

In [ ]:
# 3. Load XTTS v2 on GPU
import torch
try:
    from TTS.api import TTS
    has_gpu = torch.cuda.is_available()
    print(f"CUDA: {has_gpu}  GPU: {torch.cuda.get_device_name(0) if has_gpu else 'N/A'}")
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=has_gpu)
    print("XTTS v2 loaded")
except Exception as e:
    print(f"Failed: {e}")

In [ ]:
# 4. Start Flask server
from flask import Flask, request, send_file
import tempfile, os, threading

app = Flask(__name__)

@app.route("/health")
def health():
    return {"status": "ok", "gpu": torch.cuda.is_available() if 'torch' in dir() else False}

@app.route("/synthesize", methods=["POST"])
def synthesize():
    data = request.get_json()
    text = data.get("text", "") if data else ""
    if not text: return {"error": "text required"}, 400
    fd, path = tempfile.mkstemp(suffix=".wav")
    os.close(fd)
    tts.tts_to_file(text=text, file_path=path, speaker_wav="speaker.wav", language="en")
    return send_file(path, mimetype="audio/wav")

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False), daemon=True).start()
print("Server on :5000")

In [ ]:
# 5. Expose via Cloudflare Tunnel
import subprocess, time, re

subprocess.run(["curl", "-sL",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-o", "/usr/local/bin/cloudflared"], capture_output=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for _ in range(30):
    time.sleep(1)
    out = proc.stdout.read(4096) if proc.stdout else ""
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', out)
    if m:
        url = m.group(0)
        break

print("=" * 60)
if url:
    print(f"COLAB GPU ENDPOINT: {url}")
    print("=" * 60)
else:
    print("No URL found. Tunnel output:")
    print(proc.stdout.read()[-500:] if proc.stdout else "(none)")

while True:
    time.sleep(60)